# Training YOLOv8-Pose Hook — 100 Epoch

Model ini mendeteksi hook sekaligus 6 keypoint untuk tahap solvePnP/X-Y global.

Urutan keypoint dataset HOOKv1.2: `0=open_0, 1=point_1, 2=point_2, 3=point_3, 4=point_4, 5=hook_tip`.

In [ ]:
%pip -q install -U ultralytics
import csv, os, shutil, zipfile
from pathlib import Path
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE != 'cpu' else 'CPU'
WORKERS = os.cpu_count() or 2
print('Torch:', torch.__version__, 'Device:', DEVICE, 'GPU:', GPU_NAME, 'Workers:', WORKERS)
assert DEVICE != 'cpu', 'Training dihentikan: aktifkan Runtime > Change runtime type > T4 GPU.'

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
assert zip_names, 'Upload ZIP dataset keypoint terlebih dahulu.'
ZIP_PATH = Path('/content') / zip_names[0]
RAW_ROOT = Path('/content/hook_pose_raw')
if RAW_ROOT.exists(): shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir()
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(RAW_ROOT)
print('Dataset:', RAW_ROOT)

In [ ]:
# Cari struktur train/valid(or val)/test tanpa bergantung nama folder ZIP.
def find_split(name):
    for p in RAW_ROOT.rglob(name + '/images'):
        return p.parent
    return None
SPLITS = {'train': find_split('train'), 'val': find_split('valid') or find_split('val'), 'test': find_split('test')}
assert all(SPLITS.values()), f'Split tidak lengkap: {SPLITS}'
DATASET_ROOT = SPLITS['train'].parent
DATA_YAML = Path('/content/hook_pose.yaml')
DATA_YAML.write_text(
    f'path: {DATASET_ROOT}\n'
    'train: train/images\nval: ' + ('valid/images' if (DATASET_ROOT/'valid').exists() else 'val/images') + '\ntest: test/images\n'
    'kpt_shape: [6, 3]\n'
    'flip_idx: [0, 1, 2, 3, 4, 5]\n'
    'names:\n  0: hook\n'
)
print(DATA_YAML.read_text())

In [ ]:
# Validasi label. Format standar: class cx cy w h + 6*(x y visibility).
# Jika label hanya berisi class + 6 keypoint, bbox dibuat dari keypoint terlihat.
EXPECTED = 5 + 6 * 3
bad, counts, converted, pending_writes = [], {}, 0, {}
for split, root in SPLITS.items():
    label_dir = root / 'labels'; rows = empty = 0
    for lp in label_dir.glob('*.txt'):
        raw = lp.read_text().strip()
        if not raw: empty += 1; continue
        out = []
        for line_no, line in enumerate(raw.splitlines(), 1):
            v = line.split()
            try: nums = [float(x) for x in v]
            except ValueError: bad.append((str(lp), line_no, 'non-numeric')); continue
            if len(nums) not in (19, EXPECTED):
                bad.append((str(lp), line_no, f'field={len(nums)}, expected 19 atau {EXPECTED}')); continue
            if len(nums) == 19:
                cls, kp = nums[0], nums[1:]
                pts = [(kp[i], kp[i+1]) for i in range(0, 18, 3) if kp[i+2] > 0]
                if len(pts) < 2: bad.append((str(lp), line_no, 'keypoint terlihat < 2')); continue
                xs, ys = zip(*pts); x0, x1 = max(0,min(xs)), min(1,max(xs)); y0, y1 = max(0,min(ys)), min(1,max(ys))
                pad = 0.02; x0, x1 = max(0,x0-pad), min(1,x1+pad); y0, y1 = max(0,y0-pad), min(1,y1+pad)
                nums = [cls, (x0+x1)/2, (y0+y1)/2, x1-x0, y1-y0] + kp; converted += 1
            coord_idx = list(range(1, 5)) + [i for i in range(5, EXPECTED) if (i - 5) % 3 != 2]
            for i in coord_idx: nums[i] = min(1.0, max(0.0, nums[i]))
            coords = [nums[i] for i in coord_idx]
            if len(nums) != EXPECTED or nums[0] != 0 or any(x < 0 or x > 1 for x in coords):
                bad.append((str(lp), line_no, f'field={len(nums)} atau nilai di luar 0..1')); continue
            if any(nums[5+i] not in (0,1,2) for i in range(2, 18, 3)):
                bad.append((str(lp), line_no, 'visibility harus 0, 1, atau 2')); continue
            out.append(' '.join(f'{x:g}' for x in nums)); rows += 1
        pending_writes[lp] = '\n'.join(out) + ('\n' if out else '')
    image_files = list((root/'images').glob('*'))
    label_files = list(label_dir.glob('*.txt'))
    counts[split] = {'images': len(image_files), 'label_files': len(label_files), 'rows': rows, 'empty': empty}
assert not bad, f'Label tidak valid (contoh): {bad[:5]}'
assert all(counts[s]['rows'] > 0 for s in ('train', 'val', 'test')), f'Tidak ada label valid: {counts}'
for lp, text in pending_writes.items(): lp.write_text(text)
for cache in DATASET_ROOT.rglob('*.cache'):
    cache.unlink()
for split, root in SPLITS.items():
    verified_rows = sum(1 for lp in (root/'labels').glob('*.txt') for line in lp.read_text().splitlines() if line.strip())
    assert verified_rows > 0, f'Label tetap kosong setelah validasi: {split}'
print('counts=', counts, 'keypoint-only converted=', converted)
print('DATA YAML yang akan dipakai training:\n', DATA_YAML.read_text())

## Training

Training memakai pretrained `yolov8n-pose.pt`, 100 epoch, dan GPU Colab.

In [ ]:
model = YOLO('yolov8n-pose.pt')
results = model.train(
    data=str(DATA_YAML), epochs=100, imgsz=640, batch=-1, device=DEVICE,
    workers=WORKERS, cache=False, patience=30, project='/content/hook_pose_runs',
    name='yolov8n_hook_pose_100', exist_ok=True, pretrained=True, plots=True,
)
BEST_PT = Path('/content/hook_pose_runs/yolov8n_hook_pose_100/weights/best.pt')
assert BEST_PT.exists(), BEST_PT
print('Best model:', BEST_PT)

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=640, batch=32, device=DEVICE, workers=WORKERS, cache=False,
                         project='/content/hook_pose_runs', name='test', exist_ok=True, plots=True)
print(metrics.results_dict)
preview = sorted((SPLITS['test']/'images').glob('*'))[:12]
best_model.predict(source=[str(p) for p in preview], imgsz=640, conf=0.25, device=DEVICE,
                   save=True, project='/content/hook_pose_runs', name='preview', exist_ok=True)
print('Preview: /content/hook_pose_runs/preview')

In [ ]:
from google.colab import files
files.download(str(BEST_PT))

## Catatan

- Semua gambar harus memakai urutan 6 keypoint yang sama.
- Urutan keypoint wajib konsisten: `0=open_0`, `1=point_1`, `2=point_2`, `3=point_3`, `4=point_4`, `5=hook_tip`.
- Dataset video yang sama untuk train/test menghasilkan metrik optimistis; gunakan video berbeda untuk validasi.
- Setelah download, salin hasilnya sebagai `autonomy/vision/best_pose.pt`; model bbox lama tetap `best.pt`.